# Strict Complaint Processor — LangGraph State Machine

A structured agent workflow for processing **Downside Up** complaints using LangGraph.

## Architecture
```
intake_node → validate_node → [reject_node | categorize_node] → resolve_node → output_node
```

### Nodes
| Node | Role |
|---|---|
| `intake` | Parse & categorize the complaint |
| `validate` | Check if complaint is valid/relevant |
| `reject` | Handle invalid complaints |
| `categorize` | Deep-categorize valid complaints |
| `resolve` | Generate resolution steps |
| `output` | Format and finalize the response |

## 1. Install Dependencies

In [3]:
# Install required packages
!pip install -q langgraph langchain langchain-openai langchain-core
%pip install openai ipywidgets python-dotenv

  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)
Using cached python_dotenv-1.2.2-py3-none-any.whl (22 kB)

   ---------- ----------------------------- 1/4 [python-dotenv]
   ------------------------------ --------- 3/4 [ipywidgets]
   ------------------------------ --------- 3/4 [ipywidgets]
   ---------------------------------------- 4/4 [ipywidgets]

Note: you may need to restart the kernel to use updated packages.


## 2. Imports & Configuration

In [4]:
import os
from typing import TypedDict, Optional, List
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables from a local .env file if one exists.
load_dotenv()

class Config:
    """Application configuration."""

    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
    MODEL = os.getenv("OPENAI_MODEL", "gpt-5.2")

if not Config.OPENAI_API_KEY:
    raise ValueError(
        "Missing OPENAI_API_KEY. Set it in your environment or create a .env file with "
        "OPENAI_API_KEY=your_api_key_here"
    )

client = OpenAI(api_key=Config.OPENAI_API_KEY)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("✅ Imports complete. LLM ready.")

✅ Imports complete. LLM ready.


## 3. State Schema

All nodes read from and write to this shared `ComplaintState` TypedDict.  
LangGraph treats this as the **single source of truth** across the entire workflow.

In [5]:
class ComplaintState(TypedDict):
    """Shared state passed between every node in the graph."""

    # ── Input ──────────────────────────────────────────────────────────────────
    complaint: str                      # Raw complaint text from the user

    # ── Classification ─────────────────────────────────────────────────────────
    category: Optional[str]             # portal | monster | psychic | environmental | other
    is_valid: Optional[bool]            # True if complaint is Downside Up-related
    severity: Optional[str]             # low | medium | high | critical

    # ── Processing ─────────────────────────────────────────────────────────────
    resolution_steps: Optional[List[str]]  # Actionable steps generated by resolve_node
    rejection_reason: Optional[str]        # Why a complaint was rejected

    # ── Metadata ───────────────────────────────────────────────────────────────
    status: str                         # Current pipeline status
    workflow_path: List[str]            # Audit trail of every node visited
    final_response: Optional[str]       # Formatted output shown to the user

print("✅ ComplaintState schema defined.")

✅ ComplaintState schema defined.


## 4. Node Definitions

Each node is a **pure function** `(ComplaintState) -> ComplaintState`.  
Nodes must never mutate state in place — they return a new dict via `{**state, ...}`.

### 4.1 Intake Node

In [6]:
def intake_node(state: ComplaintState) -> ComplaintState:
    """Step 1: Intake — Parse and do an initial categorization of the complaint."""
    print("\n[INTAKE] Processing complaint...")

    complaint = state["complaint"]

    categorization_prompt = f"""Categorize this Downside Up complaint into one of these categories:
- portal: Issues with portal timing, location, or behavior
- monster: Issues with creature behavior (demogorgons, etc.)
- psychic: Issues with psychic abilities or limitations
- environmental: Issues with electricity, weather, or physical environment
- other: Anything else

Complaint: {complaint}

Respond with ONLY the category name (portal, monster, psychic, environmental, or other)."""

    response = llm.invoke([HumanMessage(content=categorization_prompt)])
    category = response.content.strip().lower()

    # Guard against unexpected LLM output
    valid_categories = {"portal", "monster", "psychic", "environmental", "other"}
    if category not in valid_categories:
        category = "other"

    new_state = {
        **state,
        "category": category,
        "workflow_path": state.get("workflow_path", []) + ["intake"],
        "status": "intake",
    }

    print(f"[INTAKE] Categorized as: {category}")
    return new_state

### 4.2 Validate Node

In [7]:
def validate_node(state: ComplaintState) -> ComplaintState:
    """
    Step 2: Validate — Decide if the complaint is genuinely Downside Up-related.

    Rule: category == 'other' triggers LLM double-check.
    All other categories are assumed valid at this stage.
    """
    print("\n[VALIDATE] Checking complaint validity...")

    complaint = state["complaint"]
    category  = state["category"]

    if category != "other":
        # Known Downside Up category → valid by definition
        is_valid = True
        print("[VALIDATE] Valid — known Downside Up category.")
    else:
        # 'other' → ask the LLM to make a final call
        validation_prompt = f"""You are a Downside Up complaint validator.
Determine if this complaint is genuinely related to the Downside Up (a supernatural parallel dimension).
Valid complaints involve: portals, creatures, psychic abilities, electromagnetic disturbances,
or other phenomena clearly linked to the Downside Up.

Complaint: {complaint}

Respond with ONLY 'valid' or 'invalid'."""

        response = llm.invoke([HumanMessage(content=validation_prompt)])
        verdict  = response.content.strip().lower()
        is_valid = (verdict == "valid")
        print(f"[VALIDATE] LLM verdict: {verdict}")

    new_state = {
        **state,
        "is_valid": is_valid,
        "workflow_path": state["workflow_path"] + ["validate"],
        "status": "validated",
    }

    print(f"[VALIDATE] Is valid: {is_valid}")
    return new_state

### 4.3 Reject Node

In [8]:
def reject_node(state: ComplaintState) -> ComplaintState:
    """
    Step 3a: Reject — Handle complaints that are not Downside Up-related.
    Sets a rejection reason and a polite final response.
    """
    print("\n[REJECT] Complaint does not meet criteria. Rejecting...")

    complaint = state["complaint"]

    rejection_prompt = f"""A user submitted a complaint that is NOT related to the Downside Up.
Write a brief, polite rejection message (2-3 sentences) explaining that this system only handles
Downside Up-related issues and suggesting they contact an appropriate authority for their actual issue.

Their complaint: {complaint}"""

    response = llm.invoke([HumanMessage(content=rejection_prompt)])
    rejection_reason = "Complaint is not related to Downside Up phenomena."

    new_state = {
        **state,
        "rejection_reason": rejection_reason,
        "final_response": f"❌ REJECTED\n\n{response.content.strip()}",
        "workflow_path": state["workflow_path"] + ["reject"],
        "status": "rejected",
    }

    print(f"[REJECT] Rejection reason: {rejection_reason}")
    return new_state

### 4.4 Categorize Node

In [9]:
def categorize_node(state: ComplaintState) -> ComplaintState:
    """
    Step 3b: Categorize — Deep analysis of valid complaints.
    Assesses severity on top of the initial category from intake_node.
    """
    print("\n[CATEGORIZE] Performing deep categorization...")

    complaint = state["complaint"]
    category  = state["category"]

    severity_prompt = f"""Assess the severity of this Downside Up complaint.
Category: {category}
Complaint: {complaint}

Severity levels:
- low: Minor inconvenience, no immediate danger
- medium: Significant disruption, potential risk
- high: Serious danger, immediate action required
- critical: Life-threatening, emergency response needed

Respond with ONLY the severity level (low, medium, high, or critical)."""

    response = llm.invoke([HumanMessage(content=severity_prompt)])
    severity = response.content.strip().lower()

    valid_severities = {"low", "medium", "high", "critical"}
    if severity not in valid_severities:
        severity = "medium"  # safe default

    new_state = {
        **state,
        "severity": severity,
        "workflow_path": state["workflow_path"] + ["categorize"],
        "status": "categorized",
    }

    print(f"[CATEGORIZE] Severity: {severity}")
    return new_state

### 4.5 Resolve Node

In [10]:
def resolve_node(state: ComplaintState) -> ComplaintState:
    """
    Step 4: Resolve — Generate concrete, actionable resolution steps
    tailored to the complaint's category and severity.
    """
    print("\n[RESOLVE] Generating resolution steps...")

    complaint = state["complaint"]
    category  = state["category"]
    severity  = state["severity"]

    resolve_prompt = f"""You are a Downside Up incident response specialist.
Generate 3-5 specific, actionable resolution steps for this complaint.

Category: {category}
Severity: {severity}
Complaint: {complaint}

Format your response as a numbered list of steps only.
Each step should be practical and specific to the Downside Up context."""

    response = llm.invoke([HumanMessage(content=resolve_prompt)])
    raw_steps = response.content.strip()

    # Parse the numbered list into a Python list
    lines = [line.strip() for line in raw_steps.split("\n") if line.strip()]
    resolution_steps = [
        line.lstrip("0123456789.-) ").strip()
        for line in lines
        if line
    ]

    new_state = {
        **state,
        "resolution_steps": resolution_steps,
        "workflow_path": state["workflow_path"] + ["resolve"],
        "status": "resolved",
    }

    print(f"[RESOLVE] Generated {len(resolution_steps)} resolution steps.")
    return new_state

### 4.6 Output Node

In [11]:
def output_node(state: ComplaintState) -> ComplaintState:
    """
    Step 5: Output — Format the final response for display.
    This is the terminal node for all valid complaints.
    """
    print("\n[OUTPUT] Formatting final response...")

    severity_emoji = {
        "low": "🟢",
        "medium": "🟡",
        "high": "🔴",
        "critical": "🚨",
    }
    emoji = severity_emoji.get(state.get("severity", "medium"), "🟡")

    steps_text = "\n".join(
        f"  {i+1}. {step}"
        for i, step in enumerate(state.get("resolution_steps", []))
    )

    final_response = f"""✅ COMPLAINT PROCESSED
{'='*55}
📋 Category : {state['category'].upper()}
{emoji} Severity  : {state.get('severity', 'N/A').upper()}
🛤️  Path      : {' → '.join(state['workflow_path'])}
{'='*55}
📝 COMPLAINT:
  {state['complaint']}

🔧 RESOLUTION STEPS:
{steps_text}
{'='*55}"""

    new_state = {
        **state,
        "final_response": final_response,
        "workflow_path": state["workflow_path"] + ["output"],
        "status": "complete",
    }

    print("[OUTPUT] Response formatted.")
    return new_state

## 5. Routing Logic (Conditional Edges)

LangGraph uses **conditional edges** to branch between nodes based on state values.  
The router function inspects the current state and returns the name of the next node.

In [12]:
def route_after_validate(state: ComplaintState) -> str:
    """
    Conditional edge: after validate_node, decide between:
      - 'reject'     → complaint is not Downside Up-related
      - 'categorize' → complaint is valid, proceed with processing
    """
    if state.get("is_valid"):
        print("[ROUTER] Valid complaint → routing to CATEGORIZE")
        return "categorize"
    else:
        print("[ROUTER] Invalid complaint → routing to REJECT")
        return "reject"

print("✅ Routing function defined.")

✅ Routing function defined.


## 6. Build the Graph

Wire up all nodes and edges into a compiled LangGraph `StateGraph`.

In [13]:
def build_complaint_graph() -> StateGraph:
    """Construct and compile the Downside Up Complaint Processor graph."""

    # ── 1. Instantiate graph with state schema ──────────────────────────────
    graph = StateGraph(ComplaintState)

    # ── 2. Register nodes ───────────────────────────────────────────────────
    graph.add_node("intake",     intake_node)
    graph.add_node("validate",   validate_node)
    graph.add_node("reject",     reject_node)
    graph.add_node("categorize", categorize_node)
    graph.add_node("resolve",    resolve_node)
    graph.add_node("output",     output_node)

    # ── 3. Set entry point ──────────────────────────────────────────────────
    graph.set_entry_point("intake")

    # ── 4. Add deterministic edges ──────────────────────────────────────────
    graph.add_edge("intake",     "validate")   # always validate after intake
    graph.add_edge("categorize", "resolve")    # always resolve after deep categorization
    graph.add_edge("resolve",    "output")     # always format output after resolution
    graph.add_edge("output",     END)          # terminal node for valid path
    graph.add_edge("reject",     END)          # terminal node for invalid path

    # ── 5. Add conditional edge (branching logic) ───────────────────────────
    graph.add_conditional_edges(
        "validate",               # source node
        route_after_validate,     # router function
        {
            "categorize": "categorize",  # router returns "categorize" → go to categorize_node
            "reject":     "reject",      # router returns "reject"     → go to reject_node
        },
    )

    # ── 6. Compile ──────────────────────────────────────────────────────────
    return graph.compile()


# Build the graph
complaint_processor = build_complaint_graph()
print("✅ Complaint processor graph compiled.")
print()
print("Graph flow:")
print("  intake → validate → [reject | categorize → resolve → output]")

✅ Complaint processor graph compiled.

Graph flow:
  intake → validate → [reject | categorize → resolve → output]


## 7. Helper: Process a Single Complaint

In [14]:
def process_complaint(complaint_text: str) -> ComplaintState:
    """
    Run a single complaint through the full LangGraph pipeline.

    Args:
        complaint_text: The raw complaint string from the user.

    Returns:
        The final ComplaintState after all nodes have run.
    """
    # Build the initial state — all optional fields start as None
    initial_state: ComplaintState = {
        "complaint":        complaint_text,
        "category":         None,
        "is_valid":         None,
        "severity":         None,
        "resolution_steps": None,
        "rejection_reason": None,
        "status":           "pending",
        "workflow_path":    [],
        "final_response":   None,
    }

    # Invoke the compiled graph — LangGraph handles the traversal
    final_state = complaint_processor.invoke(initial_state)
    return final_state

print("✅ process_complaint helper ready.")

✅ process_complaint helper ready.


## 8. Run the Test Suite

Five test cases covering all four valid categories plus one invalid complaint that **must be rejected**.

In [15]:
test_complaints = [
    "The Downside Up portal opens at different times each day. How do I predict when?",
    "Demogorgons sometimes work together and sometimes fight. What's their deal?",
    "El can move things with her mind but can't lift heavy rocks. Why?",
    "Why do creatures and power lines react so strangely together?",
    "This is not a valid complaint about something random",  # ← should be REJECTED
]

print("Testing workflow with sample complaints...\n")
print("=" * 60)

results = []
for i, complaint in enumerate(test_complaints, 1):
    print(f"\n{'='*60}")
    print(f"TEST {i}/{len(test_complaints)}")
    print(f"Complaint: {complaint}")
    print(f"{'='*60}")

    final_state = process_complaint(complaint)
    results.append(final_state)

    print("\n" + final_state["final_response"])

print("\n" + "=" * 60)
print("ALL TESTS COMPLETE")
print("=" * 60)

Testing workflow with sample complaints...


TEST 1/5
Complaint: The Downside Up portal opens at different times each day. How do I predict when?

[INTAKE] Processing complaint...
[INTAKE] Categorized as: portal

[VALIDATE] Checking complaint validity...
[VALIDATE] Valid — known Downside Up category.
[VALIDATE] Is valid: True
[ROUTER] Valid complaint → routing to CATEGORIZE

[CATEGORIZE] Performing deep categorization...
[CATEGORIZE] Severity: medium

[RESOLVE] Generating resolution steps...
[RESOLVE] Generated 5 resolution steps.

[OUTPUT] Formatting final response...
[OUTPUT] Response formatted.

✅ COMPLAINT PROCESSED
📋 Category : PORTAL
🟡 Severity  : MEDIUM
🛤️  Path      : intake → validate → categorize → resolve
📝 COMPLAINT:
  The Downside Up portal opens at different times each day. How do I predict when?

🔧 RESOLUTION STEPS:
  1. **Check Scheduled Maintenance Notifications**: Regularly review the portal's maintenance schedule posted on the Downside Up website or user dashboard to

## 9. Results Summary

In [16]:
print("\n📊 RESULTS SUMMARY")
print("=" * 60)
print(f"{'#':<4} {'Status':<12} {'Category':<16} {'Severity':<10} {'Path'}")
print("-" * 60)

for i, r in enumerate(results, 1):
    status   = r.get("status", "?").upper()
    category = (r.get("category") or "N/A").upper()
    severity = (r.get("severity") or "N/A").upper()
    path     = " → ".join(r.get("workflow_path", []))
    print(f"{i:<4} {status:<12} {category:<16} {severity:<10} {path}")

print("-" * 60)

# Success criteria check
total    = len(results)
rejected = sum(1 for r in results if r["status"] == "rejected")
resolved = sum(1 for r in results if r["status"] == "complete")

print(f"\n✅ Total processed : {total}")
print(f"✅ Valid & resolved : {resolved}")
print(f"✅ Rejected (invalid): {rejected}")

# Verify the last complaint was rejected (success criterion)
assert results[-1]["status"] == "rejected", "❌ FAIL: Last complaint should have been rejected!"
print("\n🎉 Success criteria met — invalid complaint correctly rejected!")


📊 RESULTS SUMMARY
#    Status       Category         Severity   Path
------------------------------------------------------------
1    COMPLETE     PORTAL           MEDIUM     intake → validate → categorize → resolve → output
2    COMPLETE     MONSTER          LOW        intake → validate → categorize → resolve → output
3    COMPLETE     PSYCHIC          LOW        intake → validate → categorize → resolve → output
4    COMPLETE     ENVIRONMENTAL    LOW        intake → validate → categorize → resolve → output
5    REJECTED     OTHER            N/A        intake → validate → reject
------------------------------------------------------------

✅ Total processed : 5
✅ Valid & resolved : 4
✅ Rejected (invalid): 1

🎉 Success criteria met — invalid complaint correctly rejected!


## 10. Comparison: LangGraph vs LangChain Agents.

| Aspect | **LangGraph (this notebook)** | **Freeform LangChain Agent** |
|---|---|---|
| **Workflow** | Explicit state machine with defined nodes & edges | Agent decides its own tool-calling sequence |
| **State** | Typed `ComplaintState` dict — shared, auditable | Implicit in agent's memory / scratchpad |
| **Branching** | Deterministic conditional edges (`route_after_validate`) | LLM decides whether to branch |
| **Auditability** | Full `workflow_path` trace on every run | Hard to replay or inspect |
| **Reliability** | Same graph topology every run | Can take unpredictable paths |
| **Flexibility** | Fixed structure — new steps = new nodes | Agent can improvise with available tools |
| **Best for** | Compliance, rule-based processing, SLAs | Open-ended research, creative tasks |

### When to choose LangGraph
- You need **reproducible, auditable** workflows
- Business rules must be **enforced structurally** (not just prompted)
- State needs to be **inspected or resumed** mid-flow
- You're building **production systems** with SLAs

### When to choose a freeform agent
- The task is **exploratory or creative**
- You can't pre-define all necessary steps
- You want the LLM to **self-direct** tool usage